In [1]:
import pandas as pd

df_2024 = pd.read_csv("../data/raw/atp_matches_2024.csv")
df_2025 = pd.read_csv("../data/raw/atp_matches_2025.csv")

df = pd.concat([df_2024, df_2025])

print("Matches loaded:", len(df))

Matches loaded: 6020


In [2]:
# Feature Engineering

df["rank_difference"] = (
    df["loser_rank"] -
    df["winner_rank"]
)

df["rank_points_difference"] = (
    df["winner_rank_points"] -
    df["loser_rank_points"]
)

df["age_difference"] = (
    df["winner_age"] -
    df["loser_age"]
)

print("Features created")

Features created


In [3]:
v1_df = df[
    [
        "surface",
        "tourney_level",
        "round",
        "best_of",
        "rank_difference",
        "rank_points_difference",
        "age_difference"
    ]
].copy()

print(v1_df.head())

  surface tourney_level round  best_of  rank_difference  \
0    Hard             A     F        3             -6.0   
1    Hard             A    SF        3             31.0   
2    Hard             A    SF        3             41.0   
3    Hard             A    QF        3            108.0   
4    Hard             A    QF        3              5.0   

   rank_points_difference  age_difference  
0                 -1090.0            12.0  
1                  2538.0            -5.8  
2                  1668.0             2.9  
3                  3087.0           -11.3  
4                   101.0             3.6  


In [4]:
v1_df["target"] = 1

print(v1_df.head())

  surface tourney_level round  best_of  rank_difference  \
0    Hard             A     F        3             -6.0   
1    Hard             A    SF        3             31.0   
2    Hard             A    SF        3             41.0   
3    Hard             A    QF        3            108.0   
4    Hard             A    QF        3              5.0   

   rank_points_difference  age_difference  target  
0                 -1090.0            12.0       1  
1                  2538.0            -5.8       1  
2                  1668.0             2.9       1  
3                  3087.0           -11.3       1  
4                   101.0             3.6       1  


In [5]:
print(len(v1_df))

6020


In [6]:
wins_df = v1_df.copy()

wins_df["target"] = 1

print(len(wins_df))
wins_df.head()

6020


,surface,tourney_level,round,best_of,rank_difference,rank_points_difference,age_difference,target
0,Hard,A,F,3,-6.0,-1090.0,12.0,1
1,Hard,A,SF,3,31.0,2538.0,-5.8,1
2,Hard,A,SF,3,41.0,1668.0,2.9,1
3,Hard,A,QF,3,108.0,3087.0,-11.3,1
4,Hard,A,QF,3,5.0,101.0,3.6,1


In [7]:
losses_df = v1_df.copy()

losses_df["rank_difference"] *= -1
losses_df["rank_points_difference"] *= -1
losses_df["age_difference"] *= -1

losses_df["target"] = 0

print(len(losses_df))
losses_df.head()

6020


,surface,tourney_level,round,best_of,rank_difference,rank_points_difference,age_difference,target
0,Hard,A,F,3,6.0,1090.0,-12.0,0
1,Hard,A,SF,3,-31.0,-2538.0,5.8,0
2,Hard,A,SF,3,-41.0,-1668.0,-2.9,0
3,Hard,A,QF,3,-108.0,-3087.0,11.3,0
4,Hard,A,QF,3,-5.0,-101.0,-3.6,0


In [8]:
training_df = pd.concat(
    [wins_df, losses_df],
    ignore_index=True
)

print(training_df.shape)

training_df.head()

(12040, 8)


,surface,tourney_level,round,best_of,rank_difference,rank_points_difference,age_difference,target
0,Hard,A,F,3,-6.0,-1090.0,12.0,1
1,Hard,A,SF,3,31.0,2538.0,-5.8,1
2,Hard,A,SF,3,41.0,1668.0,2.9,1
3,Hard,A,QF,3,108.0,3087.0,-11.3,1
4,Hard,A,QF,3,5.0,101.0,3.6,1


In [9]:
print(training_df["target"].value_counts())

target
1    6020
0    6020
Name: count, dtype: int64


In [10]:
print(training_df.shape)

(12040, 8)


In [11]:
print(training_df["surface"].unique())
print(training_df["tourney_level"].unique())
print(training_df["round"].unique())

<StringArray>
['Hard', 'Clay', 'Grass']
Length: 3, dtype: str
<StringArray>
['A', 'G', 'M', 'O', 'F', 'D']
Length: 6, dtype: str
<StringArray>
['F', 'SF', 'QF', 'R16', 'R32', 'RR', 'R128', 'R64', 'BR']
Length: 9, dtype: str
The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.


In [12]:
training_encoded = training_df.copy()

# Surface
surface_map = {
    "Hard": 0,
    "Clay": 1,
    "Grass": 2
}

training_encoded["surface"] = (
    training_encoded["surface"]
    .map(surface_map)
)

# Tournament Level
level_map = {
    "A": 0,
    "M": 1,
    "G": 2,
    "D": 3,
    "F": 4,
    "O": 5
}

training_encoded["tourney_level"] = (
    training_encoded["tourney_level"]
    .map(level_map)
)

# Round
round_map = {
    "R128": 1,
    "R64": 2,
    "R32": 3,
    "R16": 4,
    "QF": 5,
    "SF": 6,
    "F": 7,
    "RR": 8,
    "BR": 9
}

training_encoded["round"] = (
    training_encoded["round"]
    .map(round_map)
)

print(training_encoded.head())

   surface  tourney_level  round  best_of  rank_difference  \
0        0              0      7        3             -6.0   
1        0              0      6        3             31.0   
2        0              0      6        3             41.0   
3        0              0      5        3            108.0   
4        0              0      5        3              5.0   

   rank_points_difference  age_difference  target  
0                 -1090.0            12.0       1  
1                  2538.0            -5.8       1  
2                  1668.0             2.9       1  
3                  3087.0           -11.3       1  
4                   101.0             3.6       1  


In [13]:
print(training_encoded.isnull().sum())

surface                     0
tourney_level               0
round                       0
best_of                     0
rank_difference           210
rank_points_difference    210
age_difference             20
target                      0
dtype: int64


In [14]:
training_clean = training_encoded.dropna()

print("Before:", len(training_encoded))
print("After :", len(training_clean))
print("Rows Removed:", len(training_encoded) - len(training_clean))

print("\n")

print(training_clean.isnull().sum())

Before: 12040
After : 11828
Rows Removed: 212


surface                   0
tourney_level             0
round                     0
best_of                   0
rank_difference           0
rank_points_difference    0
age_difference            0
target                    0
dtype: int64


In [16]:
from sklearn.model_selection import train_test_split

X = training_clean.drop("target", axis=1)
y = training_clean["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows :", len(X_test))

Training rows: 9462
Testing rows : 2366


In [17]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000
)

model.fit(X_train, y_train)

print("Model trained successfully!")

Model trained successfully!


In [18]:
from sklearn.metrics import accuracy_score

predictions = model.predict(X_test)

accuracy = accuracy_score(
    y_test,
    predictions
)

print(f"Accuracy: {accuracy:.2%}")

Accuracy: 64.84%


In [19]:
feature_importance = pd.DataFrame({
    "feature": X.columns,
    "coefficient": model.coef_[0]
})

feature_importance = feature_importance.sort_values(
    by="coefficient",
    ascending=False
)

print(feature_importance)

                  feature  coefficient
1           tourney_level     0.007324
4         rank_difference     0.001246
3                 best_of     0.001245
5  rank_points_difference     0.000320
2                   round    -0.003208
0                 surface    -0.013152
6          age_difference    -0.016997


In [20]:
probabilities = model.predict_proba(X_test)

print(probabilities[:10])

[[0.73762642 0.26237358]
 [0.53141989 0.46858011]
 [0.43774858 0.56225142]
 [0.24584484 0.75415516]
 [0.46031073 0.53968927]
 [0.46480791 0.53519209]
 [0.55436173 0.44563827]
 [0.35689984 0.64310016]
 [0.74076326 0.25923674]
 [0.3764787  0.6235213 ]]


In [21]:
predictions = model.predict(X_test)

comparison = pd.DataFrame({
    "actual": y_test,
    "predicted": predictions
})

print(comparison.head(20))

       actual  predicted
8283        0          0
902         1          0
8189        0          1
7736        0          1
2822        1          1
5923        1          1
10926       0          0
1938        1          1
10339       0          0
7440        0          1
2260        1          1
6429        0          0
5611        1          0
3333        1          1
10627       0          0
7980        0          0
9686        0          0
3795        1          1
575         1          0
387         1          1


In [22]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_test,
    predictions
)

print(cm)

[[750 407]
 [425 784]]


In [23]:
print("AcePredict V1 Baseline")

print(f"Accuracy: {accuracy:.2%}")

print("\nConfusion Matrix:")
print(cm)

AcePredict V1 Baseline
Accuracy: 64.84%

Confusion Matrix:
[[750 407]
 [425 784]]


In [25]:
def predict_match(
    surface,
    tourney_level,
    round_,
    best_of,
    rank_difference,
    rank_points_difference,
    age_difference
):

    match_data = pd.DataFrame({
        "surface": [surface],
        "tourney_level": [tourney_level],
        "round": [round_],
        "best_of": [best_of],
        "rank_difference": [rank_difference],
        "rank_points_difference": [rank_points_difference],
        "age_difference": [age_difference]
    })

    probability = model.predict_proba(match_data)[0][1]

    print(f"Player A Win Probability: {probability:.2%}")
    print(f"Player B Win Probability: {(1-probability):.2%}")

In [26]:
predict_match(
    surface=0,
    tourney_level=2,
    round_=6,
    best_of=5,
    rank_difference=20,
    rank_points_difference=2000,
    age_difference=-2
)

Player A Win Probability: 66.84%
Player B Win Probability: 33.16%


In [27]:
def predict_match(
    surface,
    tourney_level,
    round_,
    best_of,
    rank_difference,
    rank_points_difference,
    age_difference
):

    match_data = pd.DataFrame({
        "surface": [surface],
        "tourney_level": [tourney_level],
        "round": [round_],
        "best_of": [best_of],
        "rank_difference": [rank_difference],
        "rank_points_difference": [rank_points_difference],
        "age_difference": [age_difference]
    })

    probability = model.predict_proba(match_data)[0][1]

    return probability

In [28]:
probability = predict_match(
    surface=0,
    tourney_level=2,
    round_=6,
    best_of=5,
    rank_difference=20,
    rank_points_difference=2000,
    age_difference=-2
)

print(probability)

0.668391303706237


In [29]:
# Building Real Match Predictions

In [30]:
df[[
    "winner_name",
    "winner_rank",
    "winner_rank_points",
    "winner_age"
]].head()

,winner_name,winner_rank,winner_rank_points,winner_age
0,Grigor Dimitrov,14.0,2570.0,32.6
1,Holger Rune,8.0,3660.0,20.6
2,Grigor Dimitrov,14.0,2570.0,32.6
3,Holger Rune,8.0,3660.0,20.6
4,Roman Safiullin,39.0,1122.0,26.4


In [31]:
player_stats = pd.concat([
    df[[
        "winner_name",
        "winner_rank",
        "winner_rank_points",
        "winner_age"
    ]].rename(columns={
        "winner_name": "player_name",
        "winner_rank": "rank",
        "winner_rank_points": "rank_points",
        "winner_age": "age"
    }),

    df[[
        "loser_name",
        "loser_rank",
        "loser_rank_points",
        "loser_age"
    ]].rename(columns={
        "loser_name": "player_name",
        "loser_rank": "rank",
        "loser_rank_points": "rank_points",
        "loser_age": "age"
    })
])

player_stats.head()

,player_name,rank,rank_points,age
0,Grigor Dimitrov,14.0,2570.0,32.6
1,Holger Rune,8.0,3660.0,20.6
2,Grigor Dimitrov,14.0,2570.0,32.6
3,Holger Rune,8.0,3660.0,20.6
4,Roman Safiullin,39.0,1122.0,26.4


In [32]:
latest_players = (
    player_stats
    .dropna(subset=["rank"])
    .sort_values("rank")
    .drop_duplicates(
        subset=["player_name"],
        keep="first"
    )
)

print(latest_players.shape)
latest_players.head()

(515, 4)


,player_name,rank,rank_points,age
1663,Jannik Sinner,1.0,9890.0,22.8
903,Novak Djokovic,1.0,9725.0,36.8
2268,Carlos Alcaraz,1.0,11540.0,22.3
694,Alexander Zverev,2.0,7945.0,27.9
263,Daniil Medvedev,3.0,7555.0,27.9


In [33]:
print(df.columns.tolist())

['tourney_id', 'tourney_name', 'surface', 'draw_size', 'tourney_level', 'tourney_date', 'match_num', 'winner_id', 'winner_seed', 'winner_entry', 'winner_name', 'winner_hand', 'winner_ht', 'winner_ioc', 'winner_age', 'loser_id', 'loser_seed', 'loser_entry', 'loser_name', 'loser_hand', 'loser_ht', 'loser_ioc', 'loser_age', 'score', 'best_of', 'round', 'minutes', 'w_ace', 'w_df', 'w_svpt', 'w_1stIn', 'w_1stWon', 'w_2ndWon', 'w_SvGms', 'w_bpSaved', 'w_bpFaced', 'l_ace', 'l_df', 'l_svpt', 'l_1stIn', 'l_1stWon', 'l_2ndWon', 'l_SvGms', 'l_bpSaved', 'l_bpFaced', 'winner_rank', 'winner_rank_points', 'loser_rank', 'loser_rank_points', 'rank_difference', 'rank_points_difference', 'age_difference']


In [34]:
players_winners = df[[
    "tourney_date",
    "winner_name",
    "winner_rank",
    "winner_rank_points",
    "winner_age"
]].rename(columns={
    "winner_name": "player_name",
    "winner_rank": "rank",
    "winner_rank_points": "rank_points",
    "winner_age": "age"
})

players_losers = df[[
    "tourney_date",
    "loser_name",
    "loser_rank",
    "loser_rank_points",
    "loser_age"
]].rename(columns={
    "loser_name": "player_name",
    "loser_rank": "rank",
    "loser_rank_points": "rank_points",
    "loser_age": "age"
})

players = pd.concat([
    players_winners,
    players_losers
])

players.head()

,tourney_date,player_name,rank,rank_points,age
0,20240101,Grigor Dimitrov,14.0,2570.0,32.6
1,20240101,Holger Rune,8.0,3660.0,20.6
2,20240101,Grigor Dimitrov,14.0,2570.0,32.6
3,20240101,Holger Rune,8.0,3660.0,20.6
4,20240101,Roman Safiullin,39.0,1122.0,26.4


In [35]:
print(players.shape)

(12040, 5)


In [36]:
players["tourney_date"] = pd.to_datetime(
    players["tourney_date"],
    format="%Y%m%d"
)

latest_players = (
    players
    .sort_values("tourney_date")
    .drop_duplicates(
        subset="player_name",
        keep="last"
    )
)

print(latest_players.shape)

latest_players.head()

(584, 5)


,tourney_date,player_name,rank,rank_points,age
80,2024-01-01,Chak Lam Coleman Wong,NaN,NaN,19.5
43,2024-01-01,Steven Diez,314.0,170.0,32.7
148,2024-01-15,Dane Sweeny,257.0,228.0,22.9
155,2024-01-15,Jason Kubler,113.0,561.0,30.6
171,2024-01-15,Jiri Vesely,293.0,188.0,30.5


In [37]:
latest_players.tail()

,tourney_date,player_name,rank,rank_points,age
2682,2025-12-17,Learner Tien,28.0,1550.0,20.0
2680,2025-12-17,Martin Landaluce,134.0,455.0,19.9
2679,2025-12-17,Nicolai Budkov Kjaer,136.0,450.0,19.2
2691,2025-12-17,Justin Engel,187.0,316.0,18.2
2689,2025-12-17,Nishesh Basavareddy,167.0,349.0,20.6


In [38]:
latest_players[
    latest_players["player_name"] == "Carlos Alcaraz"
]

,tourney_date,player_name,rank,rank_points,age
2667,2025-11-09,Carlos Alcaraz,1.0,11050.0,22.5


In [39]:
latest_players[
    latest_players["player_name"] == "Alexander Zverev"
]

,tourney_date,player_name,rank,rank_points,age
2697,2025-11-22,Alexander Zverev,3.0,5160.0,28.5


In [40]:
def get_player_stats(player_name):

    player = latest_players[
        latest_players["player_name"] == player_name
    ]

    if len(player) == 0:
        print("Player not found")
        return None

    return player.iloc[0]

In [41]:
alcaraz = get_player_stats("Carlos Alcaraz")

print(alcaraz)

tourney_date    2025-11-09 00:00:00
player_name          Carlos Alcaraz
rank                            1.0
rank_points                 11050.0
age                            22.5
Name: 2667, dtype: object


In [44]:
def predict_players(player_a, player_b):

    player_a_stats = get_player_stats(player_a)
    player_b_stats = get_player_stats(player_b)

    if player_a_stats is None or player_b_stats is None:
        return

    rank_difference = (
        player_b_stats["rank"]
        - player_a_stats["rank"]
    )

    rank_points_difference = (
        player_a_stats["rank_points"]
        - player_b_stats["rank_points"]
    )

    age_difference = (
        player_a_stats["age"]
        - player_b_stats["age"]
    )

    match_data = pd.DataFrame({
        "surface": [0],
        "tourney_level": [2],
        "round": [6],
        "best_of": [5],
        "rank_difference": [rank_difference],
        "rank_points_difference": [rank_points_difference],
        "age_difference": [age_difference]
    })

    probability = model.predict_proba(match_data)[0][1]

    print(f"{player_a}: {probability:.2%}")
    print(f"{player_b}: {(1 - probability):.2%}")

In [45]:
predict_players(
    "Carlos Alcaraz",
    "Alexander Zverev"
)

Carlos Alcaraz: 87.98%
Alexander Zverev: 12.02%


In [46]:
# Saving AcePredict V1 Model

In [51]:
import joblib

In [52]:
import os

os.makedirs(
    "../models",
    exist_ok=True
)

In [53]:
import joblib

joblib.dump(
    model,
    "../models/acepredict_v1.pkl"
)

['../models/acepredict_v1.pkl']

In [54]:
import os

os.makedirs(
    "../data/processed",
    exist_ok=True
)

latest_players.to_csv(
    "../data/processed/latest_players.csv",
    index=False
)

print("Player database saved!")

Player database saved!
